# Initiation à l'API Berserk

L'objectif de ce notebook est de comprendre comment utiliser l'API Berserk.

## 1. Imports de libraries

In [1]:
from dotenv import load_dotenv

import os
import sys
sys.path.append(os.path.abspath(".."))

from pprint import pprint

import berserk
import pandas as pd 

## 2. Chargement du token

In [2]:
load_dotenv("../.env")

token = os.getenv("LICHESS_API_TOKEN")

if not token:
    raise ValueError("LICHESS_API_TOKEN introuvable dans le fichier .env")

print("Token chargé avec succès")

Token chargé avec succès


## 3. Connexion à l'API Berserk

In [3]:
session = berserk.TokenSession(token)
client = berserk.Client(session=session)

print("Client Berserk créé avec succès")

Client Berserk créé avec succès


## 4. Affichage des informations de mon profil Lichess

In [4]:
account = client.account.get()
pprint(account)

{'blocking': False,
 'count': {'all': 1,
           'bookmark': 0,
           'draw': 0,
           'import': 0,
           'loss': 0,
           'me': 0,
           'playing': 0,
           'rated': 1,
           'win': 1},
 'createdAt': datetime.datetime(2026, 3, 17, 18, 3, 37, 330000, tzinfo=datetime.timezone.utc),
 'followable': True,
 'following': False,
 'id': 'raydennnnnnnnn',
 'perfs': {'blitz': {'games': 1,
                     'prog': 0,
                     'prov': True,
                     'rating': 1674,
                     'rd': 296},
           'bullet': {'games': 0,
                      'prog': 0,
                      'prov': True,
                      'rating': 1500,
                      'rd': 500},
           'classical': {'games': 0,
                         'prog': 0,
                         'prov': True,
                         'rating': 1500,
                         'rd': 500},
           'correspondence': {'games': 0,
                              'prog'

## 5. Importation des parties d'un joueur

Dans cet exemple, il est importé les parties du joueur Chesstam. Un maximim de 5 parties sont importées. Seules des parties jouées en mode rapide sont considérées.

In [5]:
username = "Chesstam"

games = client.games.export_by_player(
    username,
    max=20,
    perf_type="rapid",
    opening=True,
)

games = list(games)

print(f"{len(games)} parties récupérées")

20 parties récupérées


## 6. Inspection d'une partie

Cette première commande permet d'extraire toutes les informations d'une partie. L'API a converti le format PGN en JSON.

In [6]:
pprint(games[0])

{'clock': {'increment': 0, 'initial': 600, 'totalTime': 600},
 'createdAt': datetime.datetime(2026, 3, 18, 16, 0, 1, 371000, tzinfo=datetime.timezone.utc),
 'id': 'CQFDW7BD',
 'lastMoveAt': datetime.datetime(2026, 3, 18, 16, 5, 20, 128000, tzinfo=datetime.timezone.utc),
 'moves': 'e4 d5 Nc3 d4 Nd5 Qd6 Bc4 e6 e5 Qxe5+ Qe2 Qxe2+ Nxe2 exd5 Bxd5 Nf6 '
          'Bc4 Bc5 Nf4 a6 d3 Nc6 O-O O-O Re1 g6 Bd2 Bf5 b4 Nxb4 Bxb4 Bxb4 Rab1 '
          'Bxe1 Ra1 Bc3 Re1 Bxe1',
 'opening': {'eco': 'B01', 'name': 'Scandinavian Defense', 'ply': 2},
 'perf': 'rapid',
 'players': {'black': {'rating': 1379,
                       'ratingDiff': 5,
                       'user': {'id': 'chesstam', 'name': 'Chesstam'}},
             'white': {'rating': 1318,
                       'ratingDiff': -5,
                       'user': {'id': 'vale010101', 'name': 'vale010101'}}},
 'rated': True,
 'source': 'pool',
 'speed': 'rapid',
 'status': 'resign',
 'variant': 'standard',
 'winner': 'black'}


Cette seconde commande permet d'extraire les différents champs d'une partie.

In [7]:
print(sorted(games[0].keys()))

for key in sorted(games[0].keys()):
    print(key)

['clock', 'createdAt', 'id', 'lastMoveAt', 'moves', 'opening', 'perf', 'players', 'rated', 'source', 'speed', 'status', 'variant', 'winner']
clock
createdAt
id
lastMoveAt
moves
opening
perf
players
rated
source
speed
status
variant
winner


Les prochaines commandes permettent d'afficher les sous-champs des champs "players" et "opening".

In [8]:
for key in sorted(games[0]["players"].keys()):
    print(key)

print("\n")

for key in sorted(games[0]["players"]["white"].keys()):
    print(key)

print("\n")

for key in sorted(games[0]["players"]["white"]["user"].keys()):
    print(key)

print("\n")

black
white


rating
ratingDiff
user


id
name




In [9]:
for key in sorted(games[0]["opening"].keys()):
    print(key)

eco
name
ply


# 7. Applatissement d'une partie

La commande suivante éxécute une fonction permettant de convertir una partie brute en une structure plate.

In [10]:
from src.ingestion.flatten_games import flatten_game

flat = flatten_game(games[0])
flat

{'game_id': 'CQFDW7BD',
 'datetime_utc': datetime.datetime(2026, 3, 18, 16, 0, 1, 371000, tzinfo=datetime.timezone.utc),
 'perf': 'rapid',
 'speed': 'rapid',
 'rated': True,
 'source': 'pool',
 'status': 'resign',
 'winner': 'black',
 'white_id': 'vale010101',
 'white_name': 'vale010101',
 'black_id': 'chesstam',
 'black_name': 'Chesstam',
 'white_rating_before': 1318,
 'black_rating_before': 1379,
 'white_rating_diff': -5,
 'black_rating_diff': 5,
 'white_rating_after': 1313,
 'black_rating_after': 1384,
 'opening_eco': 'B01',
 'opening_name': 'Scandinavian Defense',
 'clock_initial': 600,
 'clock_increment': 0,
 'has_increment': 0,
 'moves': 'e4 d5 Nc3 d4 Nd5 Qd6 Bc4 e6 e5 Qxe5+ Qe2 Qxe2+ Nxe2 exd5 Bxd5 Nf6 Bc4 Bc5 Nf4 a6 d3 Nc6 O-O O-O Re1 g6 Bd2 Bf5 b4 Nxb4 Bxb4 Bxb4 Rab1 Bxe1 Ra1 Bc3 Re1 Bxe1',
 'ply_count': 38}

# 8. Conversion en une partie du point de vue du joueur

La commande suivante fait appel à une fonction permettant de transformer une partie aplatie en 2 rangées (1 par joueur).

In [11]:
from src.ingestion.player_view import game_to_player_rows

flat = flatten_game(games[1])
rows = game_to_player_rows(flat)

rows

[{'game_id': 'fcdmxSJ1',
  'datetime_utc': datetime.datetime(2026, 3, 18, 10, 49, 51, 647000, tzinfo=datetime.timezone.utc),
  'weekday': 2,
  'player_id': 'chesstam',
  'player_name': 'Chesstam',
  'opponent_id': 'jgiji',
  'opponent_name': 'JGIJI',
  'color_player': 'white',
  'result_player': 0.0,
  'elo_player_before': 1383,
  'elo_player_after': 1379,
  'elo_diff_player': -4,
  'termination_type': 'resign',
  'has_increment': 0,
  'opening_family': 'Rapport-Jobava System',
  'opening_eco': 'D01',
  'opening_name': 'Rapport-Jobava System',
  'ply_count': 76,
  'perf': 'rapid',
  'speed': 'rapid',
  'rated': True,
  'source': 'pool'},
 {'game_id': 'fcdmxSJ1',
  'datetime_utc': datetime.datetime(2026, 3, 18, 10, 49, 51, 647000, tzinfo=datetime.timezone.utc),
  'weekday': 2,
  'player_id': 'jgiji',
  'player_name': 'JGIJI',
  'opponent_id': 'chesstam',
  'opponent_name': 'Chesstam',
  'color_player': 'black',
  'result_player': 1.0,
  'elo_player_before': 1490,
  'elo_player_after': 1

# 9. Table complète des parties du joueur

La commande suivante permet de convertir un ensemble de parties brutes en un ensemble de parties aplaties en 2 rangées (format de la section précédente).

In [12]:
from src.ingestion.build_player_games import build_player_games

df_player_games = build_player_games(games)

df_player_games = df_player_games[df_player_games["player_id"] == "chesstam"].reset_index(drop=True)

print(len(games))

print(len(df_player_games))

print(df_player_games.columns)

df_player_games.head()



20
20
Index(['game_id', 'datetime_utc', 'weekday', 'player_id', 'player_name',
       'opponent_id', 'opponent_name', 'color_player', 'result_player',
       'elo_player_before', 'elo_player_after', 'elo_diff_player',
       'termination_type', 'has_increment', 'opening_family', 'opening_eco',
       'opening_name', 'ply_count', 'perf', 'speed', 'rated', 'source'],
      dtype='str')


,game_id,datetime_utc,weekday,player_id,player_name,opponent_id,opponent_name,color_player,result_player,elo_player_before,...,termination_type,has_increment,opening_family,opening_eco,opening_name,ply_count,perf,speed,rated,source
0,CQFDW7BD,2026-03-18 16:00:01.371000+00:00,2,chesstam,Chesstam,vale010101,vale010101,black,1.0,1379,...,resign,0,Scandinavian Defense,B01,Scandinavian Defense,38,rapid,rapid,True,pool
1,fcdmxSJ1,2026-03-18 10:49:51.647000+00:00,2,chesstam,Chesstam,jgiji,JGIJI,white,0.0,1383,...,resign,0,Rapport-Jobava System,D01,Rapport-Jobava System,76,rapid,rapid,True,pool
2,Y2Mfy0dv,2026-03-17 19:32:27.908000+00:00,1,chesstam,Chesstam,felixpito,Felixpito,black,1.0,1376,...,resign,0,Queen's Pawn Game,D00,Queen's Pawn Game,132,rapid,rapid,True,pool
3,tMUQMqEi,2026-03-17 19:17:08.521000+00:00,1,chesstam,Chesstam,juleskeats,juleskeats,white,0.0,1382,...,mate,0,Rapport-Jobava System,D01,Rapport-Jobava System,46,rapid,rapid,True,pool
4,G2NX03pO,2026-03-17 19:07:26.370000+00:00,1,chesstam,Chesstam,soullforged,soullforged,black,1.0,1377,...,mate,0,Scandinavian Defense,B01,Scandinavian Defense: Mieses-Kotroc Variation,56,rapid,rapid,True,pool


# 10. Implémentation des features partie "entre-parties"

In [13]:
from src.features.temporal_features import add_basic_temporal_features

df_player_games_v2 = add_basic_temporal_features(df_player_games)

df_player_games_v2[["player_id", "index", "delay_previous_game", "streak_before", "streak_after"]].head(20)


,player_id,index,delay_previous_game,streak_before,streak_after
19,chesstam,0,NaN,NaN,-1
18,chesstam,1,475.919,-1.0,-2
17,chesstam,2,6723.332,-2.0,1
16,chesstam,3,840.960,1.0,-1
15,chesstam,4,602.829,-1.0,-2
14,chesstam,5,1000.346,-2.0,1
13,chesstam,6,497.326,1.0,2
12,chesstam,7,2057.259,2.0,-1
11,chesstam,8,1622.494,-1.0,-2
10,chesstam,9,580.248,-2.0,1


# 11 Création des séries temporelles

## 11.1 Série temporelle session

L'expression ci-dessous fait appel à une fonction permettant d'associer chaque partie à une session.

In [14]:
from src.features.session_features import _add_sessions

df_player_games_v3 = _add_sessions(df_player_games_v2)

df_player_games_v3[["player_id", "datetime_utc", "delay_previous_game", "session_id"]].head(30)

,player_id,datetime_utc,delay_previous_game,session_id
19,chesstam,2026-03-16 17:47:18.947000+00:00,NaN,1
18,chesstam,2026-03-16 17:55:14.866000+00:00,475.919,1
17,chesstam,2026-03-16 19:47:18.198000+00:00,6723.332,2
16,chesstam,2026-03-16 20:01:19.158000+00:00,840.960,2
15,chesstam,2026-03-16 20:11:21.987000+00:00,602.829,2
14,chesstam,2026-03-16 20:28:02.333000+00:00,1000.346,2
13,chesstam,2026-03-16 20:36:19.659000+00:00,497.326,2
12,chesstam,2026-03-16 21:10:36.918000+00:00,2057.259,2
11,chesstam,2026-03-16 21:37:39.412000+00:00,1622.494,2
10,chesstam,2026-03-16 21:47:19.660000+00:00,580.248,2


L'expression ci-dessous réplique le bloc précédent et intègre des features associées aux sessions.

In [26]:
from src.features.session_features import add_sessions

df_player_games_v4 = add_sessions(df_player_games_v2)

df_player_games_v4[["player_id", "session_id", "session_position", "session_size", "session_score", "datetime_utc", "session_delay", "session_discrete_delay"]].head(30)

,player_id,session_id,session_position,session_size,session_score,datetime_utc,session_delay,session_discrete_delay
19,chesstam,1,0,2,0.00,2026-03-16 17:47:18.947000+00:00,NaN,NaN
18,chesstam,1,1,2,0.00,2026-03-16 17:55:14.866000+00:00,NaN,NaN
17,chesstam,2,0,8,0.50,2026-03-16 19:47:18.198000+00:00,6723.332,A
16,chesstam,2,1,8,0.50,2026-03-16 20:01:19.158000+00:00,6723.332,A
15,chesstam,2,2,8,0.50,2026-03-16 20:11:21.987000+00:00,6723.332,A
14,chesstam,2,3,8,0.50,2026-03-16 20:28:02.333000+00:00,6723.332,A
13,chesstam,2,4,8,0.50,2026-03-16 20:36:19.659000+00:00,6723.332,A
12,chesstam,2,5,8,0.50,2026-03-16 21:10:36.918000+00:00,6723.332,A
11,chesstam,2,6,8,0.50,2026-03-16 21:37:39.412000+00:00,6723.332,A
10,chesstam,2,7,8,0.50,2026-03-16 21:47:19.660000+00:00,6723.332,A


## 11.2 Série temporelle games_per_day

Même principe que pour la série sessions sauf qu'ici une session de jeu est une journée.

In [27]:
from src.features.day_features import add_day_features

df_player_games_v5 = add_day_features(df_player_games_v4)

df_player_games_v5[["player_id", "datetime_utc", "day_date_utc", "day_n_games", "day_score", "day_position"]].head(30)

,player_id,datetime_utc,day_date_utc,day_n_games,day_score,day_position
19,chesstam,2026-03-16 17:47:18.947000+00:00,2026-03-16,10,0.40,0
18,chesstam,2026-03-16 17:55:14.866000+00:00,2026-03-16,10,0.40,1
17,chesstam,2026-03-16 19:47:18.198000+00:00,2026-03-16,10,0.40,2
16,chesstam,2026-03-16 20:01:19.158000+00:00,2026-03-16,10,0.40,3
15,chesstam,2026-03-16 20:11:21.987000+00:00,2026-03-16,10,0.40,4
14,chesstam,2026-03-16 20:28:02.333000+00:00,2026-03-16,10,0.40,5
13,chesstam,2026-03-16 20:36:19.659000+00:00,2026-03-16,10,0.40,6
12,chesstam,2026-03-16 21:10:36.918000+00:00,2026-03-16,10,0.40,7
11,chesstam,2026-03-16 21:37:39.412000+00:00,2026-03-16,10,0.40,8
10,chesstam,2026-03-16 21:47:19.660000+00:00,2026-03-16,10,0.40,9


## 11.3 Série temporelle games_per_week

Même principe que pour la série sessions sauf qu'ici une session de jeu est une semaine.

In [ ]:
from src.features.week_features import add_week_features

df_player_games_v6 = add_week_features(df_player_games_v5)

df_player_games_v6[["player_id", "datetime_utc", "week_id", "week_n_games", "week_score", "week_position"].head(30)

,player_id,datetime_utc,week_id,week_n_games,week_score,week_position,day_n_games
19,chesstam,2026-03-16 17:47:18.947000+00:00,2026-W12,20,0.55,0,10
18,chesstam,2026-03-16 17:55:14.866000+00:00,2026-W12,20,0.55,1,10
17,chesstam,2026-03-16 19:47:18.198000+00:00,2026-W12,20,0.55,2,10
16,chesstam,2026-03-16 20:01:19.158000+00:00,2026-W12,20,0.55,3,10
15,chesstam,2026-03-16 20:11:21.987000+00:00,2026-W12,20,0.55,4,10
14,chesstam,2026-03-16 20:28:02.333000+00:00,2026-W12,20,0.55,5,10
13,chesstam,2026-03-16 20:36:19.659000+00:00,2026-W12,20,0.55,6,10
12,chesstam,2026-03-16 21:10:36.918000+00:00,2026-W12,20,0.55,7,10
11,chesstam,2026-03-16 21:37:39.412000+00:00,2026-W12,20,0.55,8,10
10,chesstam,2026-03-16 21:47:19.660000+00:00,2026-W12,20,0.55,9,10


In [30]:
print(df_player_games_v6.columns)

Index(['game_id', 'datetime_utc', 'weekday', 'player_id', 'player_name',
       'opponent_id', 'opponent_name', 'color_player', 'result_player',
       'elo_player_before', 'elo_player_after', 'elo_diff_player',
       'termination_type', 'has_increment', 'opening_family', 'opening_eco',
       'opening_name', 'ply_count', 'perf', 'speed', 'rated', 'source',
       'index', 'delay_previous_game', 'streak_after', 'streak_before',
       'new_session', 'session_id', 'session_position', 'session_size',
       'session_score', 'is_session_start', 'session_delay',
       'session_discrete_delay', 'day_date_utc', 'day_n_games', 'day_score',
       'day_position', 'week_id', 'week_n_games', 'week_score',
       'week_position'],
      dtype='str')
